# OLS Conditioning Visualiser

Drag the **column scale ratio** slider to morph the loss landscape contour ellipses from near-circular (well-conditioned, ratio=1) to infinitely elongated (ill-conditioned, ratio=8000×). This is the exact problem §5 of `docs/startup-probe.md` describes for the probe's design matrix.

In [ ]:
# Copyright (c) 2026 J. Patrick Fulton
# Apache-2.0

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display
from probe_visuals.common import A_FIT, B_FIT, C_BLUE, C_RED, C_GREEN, C_GREY

## Theory

The OLS design matrix has two columns: x₁ = B·S and x₂ = B·S². At max sequence length S=8192, x₂/x₁ = S = 8192 — an ~8000× scale difference. This makes the Gram matrix G = XᵀX ill-conditioned: the loss landscape is a very elongated ellipsoid, and gradient-based solvers overshoot along the poorly-scaled axis.

The Jacobi preconditioner normalises both columns to [0,1], making the bowl near-circular.

In [ ]:
out_cond = widgets.Output()
kappa_label = widgets.HTML()

ratio_slider = widgets.FloatLogSlider(
    value=np.log10(8192),
    base=10, min=0, max=4, step=0.05,
    description="Column scale ratio:",
    style={"description_width": "160px"},
    layout=widgets.Layout(width="600px"),
    readout_format=".0f",
)


def draw_cond(ratio):
    # Build a 2-point system: x1=[1, r], x2=[1, r^2]
    # y = perfect model outputs (noiseless)
    r = max(ratio, 1.0)
    X = np.array([[1.0, 1.0], [r, r**2]])
    y = np.array([A_FIT * 1.0 + B_FIT * 1.0, A_FIT * r + B_FIT * r**2])

    # Gram matrix and its condition number
    G = X.T @ X
    eigvals = np.linalg.eigvalsh(G)
    kappa = eigvals[-1] / max(eigvals[0], 1e-12)

    # Optimum: a_opt = A_FIT, b_opt = B_FIT
    a_opt, b_opt = A_FIT, B_FIT

    # Grid over (a, b) space — scale grid to ratio so contours are visible
    a_range = max(A_FIT * 0.5, A_FIT / r * 50) if r > 1 else A_FIT * 0.5
    b_range = max(B_FIT * 2, B_FIT * r * 0.001) if r > 1 else B_FIT * 2
    a_vals = np.linspace(a_opt - a_range, a_opt + a_range, 300)
    b_vals = np.linspace(max(b_opt - b_range, 0.01), b_opt + b_range, 300)
    AA, BB = np.meshgrid(a_vals, b_vals)

    # Loss surface L(a,b) = sum_i (y_i - a*x1_i - b*x2_i)^2
    L = np.zeros_like(AA)
    for xi1, xi2, yi in zip(X[:, 0], X[:, 1], y):
        L += (yi - AA * xi1 - BB * xi2) ** 2

    # Normalised space: alpha = a / col1_norm, beta = b / col2_norm
    col1_norm = np.linalg.norm(X[:, 0])
    col2_norm = np.linalg.norm(X[:, 1])
    alpha_opt = a_opt * col1_norm
    beta_opt = b_opt * col2_norm
    alpha_vals = np.linspace(alpha_opt * 0.5, alpha_opt * 1.5, 300)
    beta_vals = np.linspace(beta_opt * 0.5, beta_opt * 1.5, 300)
    AL, BL = np.meshgrid(alpha_vals, beta_vals)
    L_norm = np.zeros_like(AL)
    for xi1, xi2, yi in zip(X[:, 0], X[:, 1], y):
        a_n = AL / col1_norm
        b_n = BL / col2_norm
        L_norm += (yi - a_n * xi1 - b_n * xi2) ** 2

    with out_cond:
        out_cond.clear_output(wait=True)
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
        fig.suptitle(
            f"OLS Loss Landscape — column scale ratio = {r:.0f}×   "
            f"κ(G) = {kappa:.2e}",
            fontsize=12,
        )

        levels = 15
        # Raw space
        cs1 = ax1.contourf(AA, BB, np.log1p(L), levels=levels, cmap="Blues")
        ax1.contour(AA, BB, np.log1p(L), levels=levels, colors="white", linewidths=0.4, alpha=0.5)
        ax1.plot(a_opt, b_opt, "*", color=C_RED, ms=12, zorder=5, label="Optimum")
        # Gradient step: steepest descent direction from a nearby starting point
        a_start = a_opt + a_range * 0.4
        b_start = b_opt + b_range * 0.4
        grad_a = 2 * sum((a_start * x1 + b_start * x2 - yi) * x1
                         for x1, x2, yi in zip(X[:, 0], X[:, 1], y))
        grad_b = 2 * sum((a_start * x1 + b_start * x2 - yi) * x2
                         for x1, x2, yi in zip(X[:, 0], X[:, 1], y))
        grad_mag = (grad_a**2 + grad_b**2) ** 0.5 + 1e-12
        step = min(a_range, b_range) * 0.5
        ax1.annotate(
            "", xy=(a_start - grad_a / grad_mag * step, b_start - grad_b / grad_mag * step),
            xytext=(a_start, b_start),
            arrowprops=dict(arrowstyle="->", color=C_GREEN, lw=2),
        )
        ax1.set_xlabel("a (bytes/token)")
        ax1.set_ylabel("b (bytes/token²)")
        ax1.set_title("Raw space (elongated at high ratio)")
        ax1.legend(fontsize=9)

        # Normalised space
        cs2 = ax2.contourf(AL, BL, np.log1p(L_norm), levels=levels, cmap="Oranges")
        ax2.contour(AL, BL, np.log1p(L_norm), levels=levels, colors="white", linewidths=0.4, alpha=0.5)
        ax2.plot(alpha_opt, beta_opt, "*", color=C_RED, ms=12, zorder=5, label="Optimum")
        ax2.set_xlabel("α = a × ‖x₁‖")
        ax2.set_ylabel("β = b × ‖x₂‖")
        ax2.set_title("Jacobi-normalised space (near-circular)")
        ax2.legend(fontsize=9)

        plt.tight_layout()
        display(fig)
        plt.close(fig)

    kappa_label.value = (
        f'<div style="font-family:monospace;font-size:14px">'
        f'<b>Column scale ratio:</b> {r:.0f}×   '
        f'<b>Condition number κ(G):</b> {kappa:.3e}'
        f'</div>'
    )


ratio_slider.observe(lambda change: draw_cond(change["new"]), names="value")

display(ratio_slider, kappa_label, out_cond)
draw_cond(ratio_slider.value)

## What to observe

- At ratio=1: near-circular contours, single gradient step lands close to optimum
- At ratio=8192 (realistic probe value): extreme elongation, gradient step overshoots by orders of magnitude
- The right panel (normalised) stays circular regardless of ratio — this is why Jacobi preconditioning works
- Watch the **condition number κ** update: it grows as ratio² (~67M at ratio=8192)